In [5]:
import httpx
import trafilatura
import json

# The 8 seed URLs selected for the Knowledge Graph (Domain: Financial News & Tech M&A)
SEED_URLS = [
    "https://news.microsoft.com/2022/01/18/microsoft-to-acquire-activision-blizzard-to-bring-the-joy-and-community-of-gaming-to-everyone-across-every-device/",
    "https://newsroom.ibm.com/2024-04-24-IBM-to-Acquire-HashiCorp-Inc-Creating-a-Comprehensive-End-to-End-Hybrid-Cloud-Platform",
    "https://www.oracle.com/news/announcement/oracle-buys-cerner-2021-12-20/",
    "https://www.amd.com/en/newsroom/press-releases/2022-2-14-amd-completes-acquisition-of-xilinx.html",
    "https://www.aboutamazon.com/news/company-news/amazon-aws-anthropic-ai",
    "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-fourth-quarter-and-fiscal-2024",
]

# Output file name 
OUTPUT_FILE = "crawler_output.jsonl"

In [3]:
def fetch_and_clean_page(url):
    """
    Fetches the webpage using httpx and extracts the main text using trafilatura.
    Returns the cleaned text or None if extraction fails.
    """
    try:
        # We use a standard User-Agent to prevent basic 403 Forbidden errors
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        response = httpx.get(url, headers=headers, follow_redirects=True, timeout=10.0)
        
        # Check if the request was successful (HTTP 200)
        if response.status_code == 200:
            # trafilatura extracts the main content and removes UI noise (HTML tags, menus, footers)
            clean_text = trafilatura.extract(response.text)
            return clean_text
        else:
            print(f"Failed to fetch {url}. Status code: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"Error processing {url}: {e}")
        return None

def is_page_useful(text, min_words=500):
    """
    Checks if the extracted text contains enough information.
    The lab requires keeping pages with more than 500 words.
    """
    if not text:
        return False
    
    word_count = len(text.split())
    return word_count >= min_words

In [4]:
print("Starting the extraction pipeline...\n")

# Open the file in write mode
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    saved_count = 0
    
    for url in SEED_URLS:
        print(f"Processing: {url}")
        
        # 1. Fetch and strip HTML
        text = fetch_and_clean_page(url)
        
        # 2. Check if the page is "useful" (> 500 words)
        if is_page_useful(text, min_words=500):
            word_count = len(text.split())
            print(f" -> SUCCESS: Extracted {word_count} words. Saving to JSONL.")
            
            # 3. Store results in JSON Lines format (mapping URL and text)
            data = {"url": url, "text": text}
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
            saved_count += 1
        else:
            # If text is None or word_count < 500
            word_count = len(text.split()) if text else 0
            print(f" -> SKIPPED: Page not useful (Only {word_count} words).")

print(f"\nPipeline finished! Successfully saved {saved_count} articles to {OUTPUT_FILE}.")

Starting the extraction pipeline...

Processing: https://news.microsoft.com/2022/01/18/microsoft-to-acquire-activision-blizzard-to-bring-the-joy-and-community-of-gaming-to-everyone-across-every-device/
 -> SUCCESS: Extracted 2056 words. Saving to JSONL.
Processing: https://newsroom.ibm.com/2024-04-24-IBM-to-Acquire-HashiCorp-Inc-Creating-a-Comprehensive-End-to-End-Hybrid-Cloud-Platform
 -> SUCCESS: Extracted 2316 words. Saving to JSONL.
Processing: https://www.oracle.com/news/announcement/oracle-buys-cerner-2021-12-20/
 -> SUCCESS: Extracted 1350 words. Saving to JSONL.
Processing: https://www.amd.com/en/newsroom/press-releases/2022-2-14-amd-completes-acquisition-of-xilinx.html
 -> SUCCESS: Extracted 977 words. Saving to JSONL.
Processing: https://www.aboutamazon.com/news/company-news/amazon-aws-anthropic-ai
 -> SUCCESS: Extracted 2660 words. Saving to JSONL.
Processing: https://openai.com/blog/openai-announces-leadership-transition
Failed to fetch https://openai.com/blog/openai-announ

# Information Extraction Part

In [7]:
import spacy
import json
import pandas as pd

# Load the advanced Transformer model 
print("Loading spaCy model 'en_core_web_trf'")
nlp = spacy.load("en_core_web_trf")

# Read the cleaned texts from the previous phase
articles = []
with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        articles.append(json.loads(line))

print(f"Loaded {len(articles)} articles from JSONL.")

Loading spaCy model 'en_core_web_trf'
Loaded 6 articles from JSONL.


In [8]:
extracted_entities = []
extracted_relations = []

# Labels we care 
TARGET_LABELS = {"ORG", "PERSON", "GPE", "DATE"}

print("Starting Information Extraction\n")

for article in articles:
    url = article["url"]
    # We take the first 3000 characters to avoid transformer memory limits 
    # while capturing the most important facts
    text = article["text"][:3000] 
    
    # Process text with spaCy
    doc = nlp(text)
    
    # NAMED ENTITY RECOGNITION
    for ent in doc.ents:
        if ent.label_ in TARGET_LABELS:
            # We clean the text to remove weird characters 
            clean_text = ent.text.replace('\n', ' ').strip()
            extracted_entities.append({
                "Entity": clean_text,
                "Type": ent.label_,
                "Source_URL": url
            })
            
    # RELATION EXTRACTION 
    # We look for: Subject, Verb, Object
    for sent in doc.sents:
        subjects = [w for w in sent if w.dep_ in ("nsubj", "nsubjpass") and w.ent_type_ in TARGET_LABELS]
        objects = [w for w in sent if w.dep_ in ("dobj", "pobj") and w.ent_type_ in TARGET_LABELS]
        
        if subjects and objects:
            subj = subjects[0]
            obj = objects[0]
            # Find the verb connecting them (the root of the subject)
            verb = subj.head
            if verb.pos_ == "VERB":
                extracted_relations.append({
                    "Subject": subj.text,
                    "Relation": verb.lemma_,
                    "Object": obj.text,
                    "Source_URL": url
                })

print(f"Extraction complete! Found {len(extracted_entities)} entities and {len(extracted_relations)} potential relations.")

Starting Information Extraction

Extraction complete! Found 246 entities and 25 potential relations.


In [10]:
# Create a DataFrame for Entities
df_entities = pd.DataFrame(extracted_entities)

# Drop duplicates so we don't have "Microsoft" 50 times for the same article
df_entities = df_entities.drop_duplicates()

# Save to CSV as requested by the lab deliverables
output_csv = "extracted_knowledge.csv"
df_entities.to_csv(output_csv, index=False, encoding='utf-8')

print(f"Saved entities to {output_csv}")

# Let's also display a few extracted relations just to see the result!
print("\nSample of Extracted Relations")
df_relations = pd.DataFrame(extracted_relations)
if not df_relations.empty:
    print(df_relations[["Subject", "Relation", "Object"]].head(10))
else:
    print("No direct relations found with this heuristic.")

Saved entities to extracted_knowledge.csv

Sample of Extracted Relations
       Subject  Relation     Object
0        Corp.  announce       Inc.
1    Microsoft   acquire   Blizzard
2    Microsoft    become    Tencent
3       Kotick  continue   Blizzard
4       Kotick       say      years
5          IBM  announce  HashiCorp
6  Corporation  announce     Cerner
7         Catz       say       year
8          AMD  announce     Xilinx
9          AMD    expect       year


### Information Extraction Results & Entity Ambiguity Analysis

Using the `en_core_web_trf` model , the pipeline successfully extracted 246 entities (primarily ORG, PERSON, and DATE) and 25 potential relations. We successfully captured highly relevant financial events, such as `Microsoft -> acquire -> Blizzard`, `IBM -> announce -> HashiCorp`, and `AMD -> announce -> Xilinx`. 

However, the heuristic dependency parsing also highlighted three notable cases of entity ambiguity:
1. `Corp. -> announce -> Inc.`: The parser extracted isolated legal suffixes rather than the full company names, demonstrating a failure in coreference resolution and entity span detection.
2. `Kotick -> say -> years`: The model misidentified the temporal marker "years" as the direct object of the verb, which distorts the actual business meaning of the sentence.
3. `Catz -> say -> year`: Similarly, linking Oracle's CEO to a generic time reference ("year") creates a technically valid grammatical triple that holds no actionable financial value for our Knowledge Graph.